In [0]:
schema = 'fifa_bi_dev.gold_schema'

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

matches = spark.read.table("fifa_bi_dev.silver_schema.silver_world_cup_matches")
tournament = spark.read.table("fifa_bi_dev.silver_schema.silver_worldcups")

all_data = tournament.alias('t').join(matches, 'tournament_name', 'left')\
            .select('t.*',
                    'date'
                    )\
            .groupBy(*[col(f"t.{c}") for c in tournament.columns]).agg(min('date').alias('start_date'), max('date').alias('end_date')).drop('attendance')
all_data.show()


In [0]:
all_data.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{schema}.dim_tournament')
print(f'table_name: dim_tournament is updated')